<a href="https://colab.research.google.com/github/lageniaestela/12demayo/blob/master/deteccionimg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opencv-python mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.7 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [2]:
import cv2
import mediapipe as mp

# Inicializar herramientas de dibujo y el modelo de rostros
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
drawing_spec = mp_drawing.DrawingSpec(thickness=1, circle_radius=1)

# Iniciar captura de video
cap = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(max_num_faces=1, refine_landmarks=True) as face_mesh:
    while cap.isOpened():
        success, image = cap.read()
        if not success: break

        # Convertir a RGB para el modelo
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(image_rgb)

        # Dibujar los landmarks si se detecta un rostro
        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                mp_drawing.draw_landmarks(
                    image=image,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_TESSELATION,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing.DrawingStyles.get_default_face_mesh_tesselation_style()
                )

        cv2.imshow('Landmarker en vivo', image)
        if cv2.waitKey(5) & 0xFF == 27: # Presiona ESC para salir
            break

cap.release()
cv2.destroyAllWindows()

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [3]:
!pip install mediapipe opencv-python


In [4]:


import cv2
import mediapipe as mp
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow
from base64 import b64decode, b64encode
import numpy as np
import PIL.Image
import io

# 2. Función de JavaScript para conectar la webcam al navegador
def video_stream():
  js = """
    var video;
    var div = document.createElement('div');
    var stream;
    var captureCanvas;

    async function initCamera() {
      video = document.createElement('video');
      video.style.display = 'block';
      stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
    }

    async function takePhoto() {
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      return canvas.toDataURL('image/jpeg', 0.8);
    }
    """
  display(IPython.display.Javascript(js))

def js_to_image(js_reply):
  image_bytes = b64decode(js_reply.split(',')[1])
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  img = cv2.imdecode(jpg_as_np, flags=1)
  return img

# 3. Inicializar MediaPipe
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1, refine_landmarks=True)
mp_drawing = mp.solutions.drawing_utils
drawing_spec = mp_drawing.DrawingSpec(thickness=1, circle_radius=1)

# 4. Ejecutar la cámara y procesar
import IPython
from google.colab import output

# Arrancamos la cámara en el navegador
video_stream()
display(IPython.display.HTML('<canvas id="canvas" width="640" height="480" style="display:none;"></canvas>'))

try:
  while True:
    # Capturar frame desde JS
    js_reply = eval_js('takePhoto()')
    if not js_reply: break

    # Convertir a formato OpenCV
    frame = js_to_image(js_reply)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Procesar Landmarks
    results = face_mesh.process(rgb_frame)

    # Dibujar los puntos si hay rostro
    if results.multi_face_landmarks:
      for face_landmarks in results.multi_face_landmarks:
        mp_drawing.draw_landmarks(
            image=frame,
            landmark_list=face_landmarks,
            connections=mp_face_mesh.FACEMESH_TESSELATION,
            connection_drawing_spec=mp_drawing.DrawingStyles.get_default_face_mesh_tesselation_style())

    # Mostrar resultado (limpiando la salida para que no se amontonen las fotos)
    output.clear_output(wait=True)
    cv2_imshow(frame)

except Exception as err:
  print(str(err))

AttributeError: module 'mediapipe' has no attribute 'solutions'